In [ ]:
# mmcv 2.x + mmseg 1.x — both pure-Python wheels, pip-installable on Python 3.12
!pip install -q mmengine
!pip install -q mmcv           # mmcv 2.x, no compilation
!pip install -q mmsegmentation # mmseg 1.x
!pip install -q timm scipy

In [ ]:
import os, sys

if not os.path.exists('/content/SegVit'):
    !git clone https://github.com/zbwxp/SegVit.git /content/SegVit

%cd /content/SegVit
sys.path.insert(0, '/content/SegVit')

In [ ]:
import sys, types, logging

# ── mmcv.runner was removed in mmcv 2.x; checkpoint loading moved to mmengine ──
import mmengine.runner as _eng_runner
_runner = types.ModuleType('mmcv.runner')
_runner.load_checkpoint = _eng_runner.load_checkpoint
sys.modules['mmcv.runner'] = _runner

# ── mmseg.models.builder: patch in BACKBONES/HEADS/etc. if missing in mmseg 1.x ──
import mmseg.models.builder as _builder
from mmseg.registry import MODELS
for _name in ('BACKBONES', 'HEADS', 'SEGMENTORS', 'LOSSES'):
    if not hasattr(_builder, _name):
        setattr(_builder, _name, MODELS)

# ── mmseg.utils.get_root_logger was removed in mmseg 1.x ──
import mmseg.utils as _mmseg_utils
if not hasattr(_mmseg_utils, 'get_root_logger'):
    _mmseg_utils.get_root_logger = lambda log_level='INFO': logging.getLogger('mmseg')

# ── Import SegViT's modules so their @register_module() decorators fire ──
# This makes VisionTransformer, ATMHead, etc. discoverable by init_model
import backbone
import decode_heads
import losses

import torch, mmseg
print(f'PyTorch:        {torch.__version__}')
print(f'MMSeg:          {mmseg.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
from huggingface_hub import hf_hub_download

os.makedirs('/content/checkpoints', exist_ok=True)
checkpoint_path = '/content/checkpoints/ade_51.3.pth'

if not os.path.exists(checkpoint_path):
    print('Downloading SegViT-Base checkpoint (ADE20K, 51.3 mIoU) ...')
    hf_hub_download(
        repo_id='Akide/SegViTv1',
        filename='ade_51.3.pth',
        local_dir='/content/checkpoints',
    )

print(f'Checkpoint ready — {os.path.getsize(checkpoint_path) / 1024**2:.0f} MB')

In [ ]:
import glob

print('Available SegViT configs:')
for p in sorted(glob.glob('configs/segvit/*.py')):
    print(' ', p)

In [ ]:
from mmseg.apis import init_model
import torch

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Running on: {device}')

# Update this path if the config listing above shows a different filename
config_file     = 'configs/segvit/segvit_vit-b_jax_640x640_160k_ade20k.py'
checkpoint_file = '/content/checkpoints/ade_51.3.pth'

model = init_model(config_file, checkpoint_file, device=device)
print('Model loaded!')

In [ ]:
from mmseg.apis import inference_model

# Download a sample indoor scene — replace img_path with your own image if preferred
img_path = '/content/sample.jpg'
if not os.path.exists(img_path):
    !wget -q -O /content/sample.jpg \
        'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Bikeroom.jpg/800px-Bikeroom.jpg'

result  = inference_model(model, img_path)

# mmseg 1.x returns a SegDataSample; the class-index map is in pred_sem_seg
seg_map = result.pred_sem_seg.data.squeeze(0).cpu().numpy()

print(f'Segmentation map shape:  {seg_map.shape}')
print(f'Unique class indices:    {sorted(set(seg_map.flatten().tolist()))}')

In [ ]:
import mmcv
import numpy as np
import matplotlib.pyplot as plt

palette = np.array(model.dataset_meta['palette'], dtype=np.uint8)
classes = model.dataset_meta['classes']

color_seg = palette[seg_map]                                    # (H, W, 3)
img_rgb   = mmcv.imread(img_path, channel_order='rgb')
blend     = (img_rgb * 0.5 + color_seg * 0.5).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_rgb);  axes[0].set_title('Input');         axes[0].axis('off')
axes[1].imshow(blend);    axes[1].set_title('Segmentation');  axes[1].axis('off')
plt.tight_layout()
plt.show()

detected = sorted(set(seg_map.flatten().tolist()))
print('Detected classes:', [classes[i] for i in detected])